In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pycaret.regression import *
import joblib

In [3]:
# Import dataset
df = pd.read_csv('Dataset_LST_Massive_20k.csv')
df.head()

,Region_Type,Latitude,Longitude,Altitude_m,Day_of_year,Ndvi,Albedo,Cloud_cover_pct,Humidity_pct,Insolation_wm2,Lst_celsius
0,Dense_Urban_Temperate,48.8722,2.2984,68,0,0.0384,0.0970,20.78,50,200,10.23
1,Dense_Urban_Temperate,48.8802,2.3543,62,0,0.0196,0.0926,41.81,50,200,9.30
2,Dense_Urban_Temperate,48.8444,2.3000,43,0,0.0535,0.1158,20.33,50,200,10.36
3,Dense_Urban_Temperate,48.8535,2.3226,40,0,0.0757,0.1049,34.26,50,200,9.87
4,Dense_Urban_Temperate,48.8202,2.3462,64,0,0.1615,0.0870,39.36,50,200,8.06


In [4]:
# Décomposition de la variable temporelle

df["day_sin"] = np.sin(
    2*np.pi*df["Day_of_year"]/365
)

df["day_cos"] = np.cos(
    2*np.pi*df["Day_of_year"]/365
)

df.drop(
    columns=["Day_of_year"],
    inplace=True
)

In [5]:
# Sélection des variables explicatives ET de la cible
colonnes_utiles = ['Latitude', 'Longitude', 'Altitude_m', 'Ndvi', 'Albedo', 
                   'Cloud_cover_pct', 'day_sin', 'day_cos', 'Lst_celsius']

# Création d'un sous-dataframe propre pour PyCaret
df_pycaret = df[colonnes_utiles]

In [6]:
# 1. Initialisation de l'environnement PyCaret (Pipeline auto)
# normalize=True applique une standardisation
# normalize_method='zscore' correspond au StandardScaler de Scikit-Learn
reg_experiment = setup(data=df_pycaret, 
                       target='Lst_celsius', 
                       normalize=True, 
                       normalize_method='zscore',
                       imputation_type='simple',
                       numeric_imputation='median',
                       train_size=0.8,
                       session_id=42)

# 2. Comparaison de tous les modèles (PyCaret affichera un beau tableau de résultats)
# On peut exclure certains modèles trop lents si besoin avec exclude=['xgboost'] par ex.
print("Comparaison des modèles en cours...")
best_model = compare_models()

print(f"\nLe meilleur modèle sélectionné est : {best_model}")

# 3. Finalisation du modèle (entraîne le meilleur modèle sur 100% des données pour la prod)
final_model = finalize_model(best_model)

,Description,Value
0,Session id,42
1,Target,Lst_celsius
2,Target type,Regression
3,Original data shape,"(16779, 9)"
4,Transformed data shape,"(16779, 9)"
5,Transformed train set shape,"(13423, 9)"
6,Transformed test set shape,"(3356, 9)"
7,Numeric features,8
8,Preprocess,True
9,Imputation type,simple


Comparaison des modèles en cours...


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lightgbm,Light Gradient Boosting Machine,1.7620,5.8818,2.4229,0.9893,0.1858,0.1661,0.6290
rf,Random Forest Regressor,1.7471,6.1114,2.4708,0.9889,0.1837,0.1531,3.1150
et,Extra Trees Regressor,1.7863,6.1628,2.4820,0.9888,0.1765,0.1569,1.5510
knn,K Neighbors Regressor,2.0685,8.0609,2.8376,0.9854,0.2070,0.1789,0.0440
dt,Decision Tree Regressor,2.3154,11.0751,3.3255,0.9800,0.2277,0.2113,0.0630
gbr,Gradient Boosting Regressor,2.6254,12.4000,3.5193,0.9775,0.2525,0.2667,0.7400
ada,AdaBoost Regressor,6.4946,62.9312,7.9309,0.8858,0.4181,0.5715,0.4160
lar,Least Angle Regression,11.9016,283.7396,16.8413,0.4856,0.7505,0.7772,0.1360
br,Bayesian Ridge,11.9018,283.7395,16.8413,0.4856,0.7507,0.7770,0.0980
ridge,Ridge Regression,11.9016,283.7396,16.8413,0.4856,0.7506,0.7772,0.1360



Le meilleur modèle sélectionné est : LGBMRegressor(n_jobs=-1, random_state=42)


In [7]:
holdout_pred = predict_model(best_model)

# Affichage des scores précis
from sklearn.metrics import r2_score, mean_squared_error

r2 = r2_score(holdout_pred['Lst_celsius'], holdout_pred['prediction_label'])
rmse = np.sqrt(mean_squared_error(holdout_pred['Lst_celsius'], holdout_pred['prediction_label']))
print(f"\n--- EFFICACITÉ DU MEILLEUR MODÈLE ({type(best_model).__name__}) ---")
print(f"Taux d'efficacité (R²) : {r2*100:.2f} %")
print(f"Marge d'erreur moyenne : {rmse:.2f} °C")

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Light Gradient Boosting Machine,1.7308,5.5865,2.3636,0.9898,0.1842,0.1513



--- EFFICACITÉ DU MEILLEUR MODÈLE (LGBMRegressor) ---
Taux d'efficacité (R²) : 98.98 %
Marge d'erreur moyenne : 2.36 °C


In [8]:
# Test avec un échantillon fictif (ex: Paris en été)
sample = pd.DataFrame([{
    "Latitude": 48.85,
    "Longitude": 2.35,
    "Altitude_m": 35,
    "Ndvi": 0.45,
    "Albedo": 0.18,
    "Cloud_cover_pct": 55,
    "day_sin": np.sin(2*np.pi*180/365),
    "day_cos": np.cos(2*np.pi*180/365)
}])

# Prédiction avec PyCaret
prediction_df = predict_model(final_model, data=sample)

# Dans PyCaret 3.x, la colonne de prédiction s'appelle 'prediction_label'
lst_pred = prediction_df['prediction_label'].iloc[0]
print(f"LST prédite : {lst_pred:.2f} °C")

# Sauvegarde du pipeline entier (PyCaret ajoute automatiquement l'extension .pkl)
save_model(final_model, "best_lst_model_pycaret")
print("Modèle sauvegardé sous le nom 'best_lst_model_pycaret.pkl'")

LST prédite : 32.46 °C
Transformation Pipeline and Model Successfully Saved
Modèle sauvegardé sous le nom 'best_lst_model_pycaret.pkl'


In [10]:
#  Création du modèle AVEC validation croisée explicite (ex: 5 folds)
# L'argument 'fold=5' force une validation croisée à 5 plis.
print("Création du modèle et exécution de la Validation Croisée...")


cv_model = create_model('lightgbm', fold=5) 
# Le résultat affiché par create_model sera un tableau montrant les scores pour CHAQUE fold (Fold 1, Fold 2... Fold 5) 
# ainsi que la moyenne (Mean) et l'écart-type (Std). Cela garantit l'exactitude !

# 4. Finalisation et Sauvegarde
# finalize_model va prendre ce modèle validé et l'entraîner sur 100% des données pour la mise en production
final_model_cv = finalize_model(cv_model)

save_model(final_model_cv, "best_lst_model_pycaret_cv")
print("Modèle final (validé) sauvegardé avec succès.")

Création du modèle et exécution de la Validation Croisée...


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,1.8344,6.5064,2.5508,0.9888,0.1856,0.1801
1,1.7669,5.8139,2.4112,0.9893,0.1936,0.1659
2,1.8209,6.3757,2.5250,0.9884,0.1966,0.2122
3,1.7535,5.8238,2.4133,0.9895,0.1808,0.1284
4,1.7559,5.7859,2.4054,0.9893,0.1857,0.1682
Mean,1.7863,6.0611,2.4611,0.9891,0.1885,0.1710
Std,0.0343,0.3132,0.0633,0.0004,0.0058,0.0269


Transformation Pipeline and Model Successfully Saved
Modèle final (validé) sauvegardé avec succès.
